# DSSATLab tutorial: crop simulations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/notebook/dssatlab_tutorial.ipynb)

Run the cells from top to bottom. Explore the DSSAT crop model through these cases:

1. **Case 1**: DSSAT's own example, exactly as it ships. Nothing to edit.
2. **Case 2**: your own crop. You fill in three small files (weather, soil, management) and run it.
3. **Case 3**: compare the first run with invented observed data.
4. **Case 4**: compare the first run with DSSAT's own measurements.
5. **Case 5**: another crop. List template crops and cultivars, and simulate soybean from scratch.
6. **Case 6**: a multi-treatment experiment. Write a three-treatment maize template (rainfed, irrigated, irrigated with nitrogen), vary treatments with experiment data, and compare yields with run_treatments.
7. **Case 7**: two fields in one experiment. Write a maize template with two fields, supply per-field weather and soil, and compare sites with run_treatments.
8. **Case 8**: many seasons. Run a two-treatment experiment over 9 weather years with controls years, and summarise yield statistics across seasons.
9. **Case 9**: a crop rotation. Run a multi-year sequence (bean, fallow, soybean) from an existing sequence FileX with controls years, and summarise yields per rotation component.
10. **Case 10**: a rotation from the FileX template. Build a crop rotation (maize, fallow, wheat, fallow) from scratch in a FileX template, and run it across cycles with controls years.
11. **Case 11**: experiment data per rotation component. Give maize and wheat their own fertilizer and irrigate maize over three cycles.
12. **Case 12**: a sweep over planting date and nitrogen. Run two planting dates by three nitrogen rates and compare the labelled Summary rows.
13. **Case 13**: a cultivar coefficient sweep. Vary maize P1 and compare anthesis, maturity and grain yield.
14. **Case 14**: photosynthesis options. Compare PHOTO C and L with initial conditions off.

You do not need to know Python. Every cell is one or two lines.

## Set up

In [ ]:
%pip install -q "dssatlab[plot,yaml] @ git+https://github.com/AbdelrahmanAmr3/dssatlab.git@v0.15.0"

The next cell builds the DSSAT program on Colab. The first time it takes several minutes; after that it takes a moment.

In [ ]:
import dssatlab as dl
dssat = dl.install()

## Case 1: DSSAT's own example

A maize experiment from Gainesville, Florida, with 6 treatments (rainfed or irrigated, low or high nitrogen). The next cell downloads DSSAT's own files for it into a folder called `case1_gainesville`.

In [ ]:
!wget -q -P case1_gainesville https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Maize/UFGA8201.MZX https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Maize/UFGA8201.MZA https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Maize/UFGA8201.MZT https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8201.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Soil/SOIL.SOL https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data/Genotype/MZCER048.CUL https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data/Genotype/MZCER048.ECO https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data/Genotype/MZCER048.SPE
!ls case1_gainesville

Run it. `dl.run` runs every treatment in the file.

In [ ]:
result1 = dl.run("case1_gainesville/UFGA8201.MZX")

Look at the results, one row per treatment:

- `HWAM`: grain yield (kg/ha)
- `CWAM`: total above-ground biomass (kg/ha)
- `MDAT`: date the crop matured
- `IRCM`: irrigation water applied (mm)

In [ ]:
import pandas as pd
pd.DataFrame(result1.summary())[["TNAM", "PDAT", "MDAT", "HWAM", "CWAM", "IRCM"]]

Leaf area index over the season, for each treatment:

In [ ]:
result1.plot("LAID");

## Case 2: your own crop

Now the same maize experiment file, but with **your** weather, soil and management. You only edit three files:

| File | What it holds |
|---|---|
| `my_weather.csv` | one row per day: temperature, rain, solar radiation |
| `my_soil.csv` | one row per soil layer |
| `my_experiment.yaml` | sowing, irrigation, fertilizer, cultivar, starting soil, simulation settings |

Two values must stay as they are, because the experiment file fixes them: the weather `station` code (`UFGA`) and the soil profile `soil_id` (`IBMZ910014`). Everything else is yours to change.

This example is summer maize at Kafr El-Sheikh, Egypt, in 2020. The next cell copies the experiment file into a new folder `case2_my_data` and downloads the example weather (real daily data from NASA POWER).

In [ ]:
!mkdir -p case2_my_data
!cp case1_gainesville/UFGA8201.MZX case1_gainesville/MZCER048.* case2_my_data/
!wget -q -P case2_my_data https://raw.githubusercontent.com/AbdelrahmanAmr3/dssatlab/master/notebook/tutorial_data/my_weather.csv
!ls case2_my_data

### Your weather

Your own weather goes in a CSV file with exactly these columns. Units are DSSAT's own: nothing is converted for you. If your data is in Excel, rename your columns to these names, use dates like `2020-05-15`, and save as CSV.

| Column | Meaning |
|---|---|
| `station` | station code, must be `UFGA` here |
| `latitude`, `longitude` | degrees |
| `elevation` | metres |
| `date` | `YYYY-MM-DD`, one row for every day, no gaps |
| `srad` | solar radiation, MJ/m2/day |
| `tmax`, `tmin` | daily maximum and minimum temperature, °C |
| `rain` | rainfall, mm |

In [ ]:
pd.read_csv("case2_my_data/my_weather.csv").head()

### Your soil

One row per soil layer. Edit the numbers and run the cell to save your changes.

In [ ]:
%%writefile case2_my_data/my_soil.csv
soil_id,salb,slro,sldr,slpf,slb,slll,sdul,ssat,srgf,sbdm,sloc
IBMZ910014,0.12,75,0.3,1.0,10,0.20,0.36,0.48,1.0,1.25,1.4
IBMZ910014,0.12,75,0.3,1.0,30,0.21,0.37,0.48,0.7,1.30,1.0
IBMZ910014,0.12,75,0.3,1.0,60,0.22,0.38,0.47,0.4,1.35,0.7
IBMZ910014,0.12,75,0.3,1.0,80,0.23,0.38,0.47,0.2,1.40,0.5
IBMZ910014,0.12,75,0.3,1.0,100,0.23,0.38,0.47,0.1,1.40,0.4

### Your management

Sowing, irrigation, fertilizer, cultivar and the starting soil. Edit anything and run the cell to save your changes.

In [ ]:
%%writefile case2_my_data/my_experiment.yaml
# My maize crop: Kafr El-Sheikh, Egypt, summer 2020 (treatment 1).
# Every date is in quotes: "YYYY-MM-DD". Leave a section out to keep what the FileX has.

treatments:
  1:
    planting:
      date: "2020-05-15"          # sowing date
      method: "S"                 # S = seed
      distribution: "R"           # R = rows
      population: 7.0             # plants per m2
      row_spacing: 70.0           # cm
      depth: 5.0                  # cm

    irrigation:                   # one block per irrigation: date, mm of water, method
      - {date: "2020-05-16", amount: 60, method: "IR001"}
      - {date: "2020-05-26", amount: 60, method: "IR001"}
      - {date: "2020-06-05", amount: 60, method: "IR001"}
      - {date: "2020-06-15", amount: 60, method: "IR001"}
      - {date: "2020-06-25", amount: 60, method: "IR001"}
      - {date: "2020-07-05", amount: 60, method: "IR001"}
      - {date: "2020-07-15", amount: 60, method: "IR001"}
      - {date: "2020-07-25", amount: 60, method: "IR001"}
      - {date: "2020-08-04", amount: 60, method: "IR001"}
      - {date: "2020-08-14", amount: 60, method: "IR001"}

    fertilizer:                   # n = kg of nitrogen per hectare; FE005 = urea, AP001 = broadcast
      - {date: "2020-05-15", material: "FE005", application: "AP001", depth: 5, n: 120}
      - {date: "2020-06-09", material: "FE005", application: "AP001", depth: 5, n: 120}

    cultivar:
      crop: "MZ"
      code: "IB0035"              # a cultivar code that exists in MZCER048.CUL

    initial_conditions:           # the soil on the start date, layer by layer
      date: "2020-05-01"
      previous_crop: "WH"         # wheat before maize
      residue_mass: 1500          # kg/ha of wheat residue
      layers:
        - {depth: 10,  water: 0.30, nh4: 1.0, no3: 6.0}
        - {depth: 30,  water: 0.32, nh4: 0.8, no3: 4.0}
        - {depth: 60,  water: 0.33, nh4: 0.5, no3: 3.0}
        - {depth: 100, water: 0.34, nh4: 0.4, no3: 2.0}

    controls:
      start_date: "2020-05-01"    # the simulation starts here; weather must cover it
      water: "Y"                  # simulate water
      nitrogen: "Y"               # simulate nitrogen
      output_interval: 1          # write daily output

### Check and run

`check()` reads all your files and lists every problem at once. It changes nothing and does not start DSSAT.

In [ ]:
sim = dl.Simulation(
    "case2_my_data/UFGA8201.MZX",
    treatment=1,
    weather="case2_my_data/my_weather.csv",
    soil="case2_my_data/my_soil.csv",
    management="case2_my_data/my_experiment.yaml",
)
problems = sim.check()

When every line says OK, run the simulation:

In [ ]:
result2 = sim.run()
pd.DataFrame(result2.summary())[["PDAT", "MDAT", "HWAM", "CWAM", "IRCM"]]

In [ ]:
result2.plot("LAID");

### When something is wrong

Here is a soil file whose `soil_id` does not match the experiment file. `check()` tells you what is wrong and how to fix it, before anything runs.

In [ ]:
%%writefile case2_my_data/bad_soil.csv
soil_id,salb,slro,sldr,slpf,slb,slll,sdul,ssat,srgf,sbdm,sloc
IBMZ910099,0.12,75,0.3,1.0,10,0.20,0.36,0.48,1.0,1.25,1.4
IBMZ910099,0.12,75,0.3,1.0,30,0.21,0.37,0.48,0.7,1.30,1.0
IBMZ910099,0.12,75,0.3,1.0,60,0.22,0.38,0.47,0.4,1.35,0.7
IBMZ910099,0.12,75,0.3,1.0,80,0.23,0.38,0.47,0.2,1.40,0.5
IBMZ910099,0.12,75,0.3,1.0,100,0.23,0.38,0.47,0.1,1.40,0.4

In [ ]:
dl.Simulation(
    "case2_my_data/UFGA8201.MZX",
    treatment=1,
    weather="case2_my_data/my_weather.csv",
    soil="case2_my_data/bad_soil.csv",
    management="case2_my_data/my_experiment.yaml",
).check()

The fix is to use the right `soil_id` (`IBMZ910014`), as in `my_soil.csv` above.

## Your turn

Change the numbers in the soil and management cells above and run again, for example irrigate less, sow later or add nitrogen, and compare `HWAM`. To use your own weather, upload your CSV to `case2_my_data` in the Colab file panel and use its name in the `weather=` line.

## Case 3: compare with what you measured

Use Case 1's `result1` to compare observed data with the Gainesville maize results. It holds all six treatments of `UFGA8201.MZX`; we only compare treatments 1 and 2 here, under scenario `base`.


**All numbers and dates below are invented for the tutorial, not real measurements.** Replace them with your own observed data when comparing your crop. `HWAM` is grain yield (kg[dm]/ha), `ADAT` is anthesis date, `MDAT` is maturity date, and `LAID` is leaf area index (m2/m2). Nothing is converted.

Leave `date` blank for Summary values. Fill it in for Plant growth on that day. Use `yyyy-mm-dd` for every date. The LAID dates below fall inside Case 1's 1982 season, from planting on February 26 to maturity around June 30.

In [ ]:
%%writefile case1_gainesville/my_observed.csv
# Invented observed data for the tutorial, not real measurements.
# HWAM: kg[dm]/ha; LAID: m2/m2; ADAT/MDAT and date: yyyy-mm-dd.
scenario,treatment,date,HWAM,ADAT,MDAT,LAID
base,1,,4500,1982-05-10,1982-06-28,
base,2,,6000,1982-05-12,1982-06-30,
base,1,1982-04-15,,,,1.8
base,1,1982-05-15,,,,3.2
base,2,1982-05-15,,,,4.0


Compare the observed data with the run. Each pair shows the observed value, simulated value and their difference (simulated minus observed). Dates in the table use DSSAT's `yyyyddd` integers; date errors are in days.

In [ ]:
ev = dl.evaluate(result1, "case1_gainesville/my_observed.csv")
ev.to_dataframe()

Look at RMSE, mean bias and Willmott's d-index for each variable. These statistics need at least two pairs; a variable with one pair still has its individual error.

In [ ]:
ev.statistics

Compare grain yield on a plot. Points on the dashed 1:1 line match exactly; points above it mean the simulated yield is higher.

In [ ]:
dl.plot_evaluation(ev, variable="HWAM");

## Case 4: DSSAT's own measurements

Reuse `result1` and the FileA/FileT downloaded from DSSAT's Maize folder in Case 1; keep their FileX beside them so short dates resolve from each treatment's `SDATE`.

In [ ]:
observed_a = dl.read_dssat_observed("case1_gainesville/UFGA8201.MZA")
observed_t = dl.read_dssat_observed("case1_gainesville/UFGA8201.MZT")

Compare the end-of-season (FileA) measurements for all six treatments; unsupported columns and unmeasured `-99` cells are not read.

In [ ]:
ev_dssat = dl.evaluate(result1, observed_a)
ev_dssat.to_dataframe()

Draw the simulated LAID curves with the FileT points in the same colour for each treatment. Some FileT dates fall outside the season; the plot still shows them, while `evaluate()` would report them as problems.

In [ ]:
dl.plot_observed(result1, observed_t, "LAID");

Read DSSAT's own evaluation: `HWAMS` is simulated yield and `HWAMM` is measured yield; Case 1 supplied the FileA before running.

In [ ]:
dssat_yield = pd.DataFrame(result1.dssat_evaluation())[["TN", "HWAMS", "HWAMM"]]
yield_pairs = ev_dssat.to_dataframe().query("variable == 'HWAM'")

Show DSSATLab's HWAM pairs beside the DSSAT evaluation, matched by treatment (`TN`); `dl.read_dssat_evaluation(result1.run_dir)` reads the same DSSAT rows.

In [ ]:
yield_pairs.merge(dssat_yield, left_on="treatment", right_on="TN")

## Case 5: another crop

List the template crops and cultivars your DSSAT install supports, then write a soybean FileX from scratch using Case 2's weather and soil.

See which template crops your installed DSSAT has genotype files for:

In [ ]:
dl.list_crops()

List available soybean cultivars and view the first few rows:

In [ ]:
pd.DataFrame(dl.list_cultivars("soybean")).head()

Write a FileX template for soybean with cultivar IB0011 (EVANS) from the list, sown on Case 2's date:

In [ ]:
%%writefile case2_my_data/my_filex.yaml
crop: "soybean"
treatment_name: "My soybean crop"
cultivar:
  code: "IB0011"
planting:
  date: "2020-05-15"
  method: "S"
  distribution: "R"
  population: 30
  row_spacing: 50
  depth: 4

Check the template against Case 2's weather and soil:

In [ ]:
sim = dl.Simulation(
    filex_template="case2_my_data/my_filex.yaml",
    weather="case2_my_data/my_weather.csv",
    soil="case2_my_data/my_soil.csv",
)
sim.check()

Run the simulation and look at the grain yield (`HWAM`). The template adds no irrigation and the summer is dry, so the yield is low; add irrigation with experiment data as in Case 2.

In [ ]:
result5 = sim.run()
pd.DataFrame(result5.summary())[["PDAT", "MDAT", "HWAM", "CWAM"]]

## Case 6: a multi-treatment experiment

Write a three-treatment maize FileX template from scratch (Rainfed, Irrigated, Irrigated + N 120) reusing Case 2's weather and soil, vary the treatments with experiment data, and compare yields with `run_treatments`.

Write a FileX template with three named treatments starting from the base sowing and cultivar:

In [ ]:
%%writefile case2_my_data/multi_filex.yaml
crop: "maize"
treatments:
  - "Rainfed"
  - "Irrigated"
  - "Irrigated + N 120"
cultivar:
  code: "IB0035"
planting:
  date: "2020-05-15"
  method: "S"
  distribution: "R"
  population: 7.0
  row_spacing: 70
  depth: 5


Write experiment data for treatments 2 and 3: Case 2's irrigation on both, plus 120 kg N on treatment 3. Treatment 1 has no entry and stays rainfed with no fertilizer:

In [ ]:
%%writefile case2_my_data/multi_experiment.yaml
treatments:
  2:
    irrigation:
      - {date: "2020-05-16", amount: 60, method: "IR001"}
      - {date: "2020-05-26", amount: 60, method: "IR001"}
      - {date: "2020-06-05", amount: 60, method: "IR001"}
      - {date: "2020-06-15", amount: 60, method: "IR001"}
      - {date: "2020-06-25", amount: 60, method: "IR001"}
      - {date: "2020-07-05", amount: 60, method: "IR001"}
      - {date: "2020-07-15", amount: 60, method: "IR001"}
      - {date: "2020-07-25", amount: 60, method: "IR001"}
      - {date: "2020-08-04", amount: 60, method: "IR001"}
      - {date: "2020-08-14", amount: 60, method: "IR001"}
  3:
    irrigation:
      - {date: "2020-05-16", amount: 60, method: "IR001"}
      - {date: "2020-05-26", amount: 60, method: "IR001"}
      - {date: "2020-06-05", amount: 60, method: "IR001"}
      - {date: "2020-06-15", amount: 60, method: "IR001"}
      - {date: "2020-06-25", amount: 60, method: "IR001"}
      - {date: "2020-07-05", amount: 60, method: "IR001"}
      - {date: "2020-07-15", amount: 60, method: "IR001"}
      - {date: "2020-07-25", amount: 60, method: "IR001"}
      - {date: "2020-08-04", amount: 60, method: "IR001"}
      - {date: "2020-08-14", amount: 60, method: "IR001"}
    fertilizer:
      - {date: "2020-05-15", material: "FE005", application: "AP001", depth: 5, n: 60}
      - {date: "2020-06-09", material: "FE005", application: "AP001", depth: 5, n: 60}


Run all treatments in one call using `run_treatments`:

In [ ]:
results6 = dl.run_treatments(
    filex_template="case2_my_data/multi_filex.yaml",
    weather="case2_my_data/my_weather.csv",
    soil="case2_my_data/my_soil.csv",
    management="case2_my_data/multi_experiment.yaml",
)


Combine the summaries and display grain yield (`HWAM`) for each treatment:

In [ ]:
pd.DataFrame(dl.combine_summaries(results6))[["treatment", "TNAM", "HWAM"]]


## Case 7: two fields in one experiment

Write a maize FileX template comparing two fields (`Site A` and `Site B`), supply per-field weather and soil, and compare sites with `run_treatments`.

Write a FileX template assigning each treatment to its own field with `treatment_fields`:

In [ ]:
%%writefile case2_my_data/two_field_filex.yaml
crop: "maize"
treatments:
  - "Site A"
  - "Site B"
treatment_fields: [1, 2]
cultivar:
  code: "IB0035"
planting:
  date: "2020-05-15"
  method: "S"
  distribution: "R"
  population: 7.0
  row_spacing: 70
  depth: 5


Field 1 reuses Case 2's weather CSV. For field 2, copy Case 2's weather rows under station code `UFGB`:

In [ ]:
weather_field2 = pd.read_csv("case2_my_data/my_weather.csv").assign(station="UFGB")


Run both fields in one call using `run_treatments` with per-field weather and soil dictionaries:

In [ ]:
results7 = dl.run_treatments(
    filex_template="case2_my_data/two_field_filex.yaml",
    weather={1: "case2_my_data/my_weather.csv", 2: weather_field2},
    soil={1: "case2_my_data/my_soil.csv", 2: "case2_my_data/my_soil.csv"},
)


Combine the summaries and display `WSTA`, `SOIL_ID`, and grain yield (`HWAM`) for each treatment:

In [ ]:
pd.DataFrame(dl.combine_summaries(results7))[["treatment", "TNAM", "WSTA", "SOIL_ID", "HWAM"]]


## Case 8: many seasons

Run a two-treatment maize experiment over 9 weather years with controls `years`, and summarise yield statistics across seasons with `summarize_seasons`.

Download 10 years of Gainesville weather files (1978–1987) from DSSAT's data repository:

In [ ]:
!wget -q -P case8_seasonal https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA7801.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA7901.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8001.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8101.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8201.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8301.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8401.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8501.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8601.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8701.WTH


Read the weather files into one continuous list of daily weather rows:

In [ ]:
from datetime import date, timedelta
from pathlib import Path

def weather_rows(name):
    rows = []
    lines = Path(f"case8_seasonal/{name}.WTH").read_text().splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith("@ INSI"))
    insi, lat, lon, elev, tav, amp = (lines[i + 1].split() + ["-99"] * 6)[:6]
    hdr = next(k for k, l in enumerate(lines) if l.startswith("@DATE"))
    for l in lines[hdr + 1:]:
        if not l.strip() or not l[:1].isdigit():
            continue
        p = l.split()
        d = date(1900 + int(p[0][:2]), 1, 1) + timedelta(days=int(p[0][2:]) - 1)
        rows.append(dict(station=insi, latitude=float(lat), longitude=float(lon),
                         elevation=float(elev), date=d.isoformat(), srad=float(p[1]),
                         tmax=float(p[2]), tmin=float(p[3]), rain=float(p[4]),
                         tav=float(tav), amp=float(amp), refht=2.0, wndht=2.0))
    return rows

weather8 = [r for y in range(78, 88) for r in weather_rows(f"UFGA{y}01")]
weather8 = [dict(r, tav=weather8[0]["tav"], amp=weather8[0]["amp"]) for r in weather8]


Write a two-treatment maize FileX template starting from 1978-03-01:

In [ ]:
%%writefile case8_seasonal/seasonal_filex.yaml
crop: "maize"
treatments:
  - "Low N"
  - "High N"
cultivar:
  code: "IB0035"
planting:
  date: "1978-03-01"
  method: "S"
  distribution: "R"
  population: 7.2
  row_spacing: 75
  depth: 5


Write experiment data setting controls `years` to 9 for both treatments, and 120 kg N for treatment 2:

In [ ]:
%%writefile case8_seasonal/seasonal_experiment.yaml
treatments:
  1:
    controls:
      years: 9
  2:
    controls:
      years: 9
    fertilizer:
      - date: "1978-03-01"
        material: "FE005"
        application: "AP002"
        depth: 5
        n: 120


Run both treatments over 9 seasons in one call using `run_treatments`:

In [ ]:
results8 = dl.run_treatments(
    filex_template="case8_seasonal/seasonal_filex.yaml",
    weather=weather8,
    soil="case2_my_data/my_soil.csv",
    management="case8_seasonal/seasonal_experiment.yaml",
)


Summarise grain yield (`HWAM`) across all 9 seasons with `summarize_seasons`:

In [ ]:
dl.to_dataframe(dl.summarize_seasons(dl.combine_summaries(results8)))


## Case 9: a crop rotation

Run a multi-year sequence (dry bean, fallow, soybean) from an existing sequence FileX with controls `years`, and summarise yields per rotation component.

Download DSSAT's sequence experiment file and soil library into a folder called `case9_sequence`:

In [ ]:
!wget -q -P case9_sequence https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Sequence/UFGA7804.SQX https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Soil/SOIL.SOL


Fix the copied FileX settings (WTHER W -> M, FNAME Y -> N, NREPS 5 -> 1) so it uses your weather and standard outputs:

In [ ]:
from pathlib import Path

sqx = Path("case9_sequence/UFGA7804.SQX")
text = sqx.read_text(encoding="latin-1")
text = text.replace(" 1 GE             10     5 ", " 1 GE             10     1 ")
text = text.replace(" ME              W ", " ME              M ")
text = text.replace(" OU              Y ", " OU              N ")
sqx.write_text(text, encoding="latin-1")


Create a Simulation for treatment 1 with controls `years: 9`, reusing Case 8's Gainesville weather rows (1978–1987):

In [ ]:
sim9 = dl.Simulation(
    "case9_sequence/UFGA7804.SQX",
    treatment=1,
    weather=weather8,
    management={"treatments": {1: {"controls": {"years": 9}}}},
)


Check inputs before running. The report identifies treatment 1 as a sequence of 6 rotation components:

In [ ]:
sim9.check()


Run the sequence simulation:

In [ ]:
result9 = sim9.run()


Inspect the summary results, showing rotation component (`R#`), crop (`CR`), and yield (`HWAM`):

In [ ]:
pd.DataFrame(result9.summary())[["R#", "CR", "PDAT", "MDAT", "HWAM"]]


Summarise yield statistics per rotation component across the 9 years with `summarize_seasons`:

In [ ]:
dl.to_dataframe(dl.summarize_seasons(result9.summary()))


## Case 10: a rotation from the FileX template

Build a 4-component crop rotation (maize, fallow, wheat, fallow) from scratch using a FileX template, run it with your own weather and soil data, and explore multi-year cycles with controls `years`.

Extract the IBSB910015 soil profile downloaded in Case 9 into soil rows:

In [ ]:
lines = Path("case9_sequence/SOIL.SOL").read_text(encoding="latin-1").splitlines()
start = next(i for i, l in enumerate(lines) if l.startswith("*IBSB910015"))
p = next(l for l in lines[start:] if l.strip().startswith("-99")).split()
h_idx = next(start + k for k, l in enumerate(lines[start:]) if "@  SLB" in l or "@ SLB" in l)
soil10 = []
for l in lines[h_idx + 1:]:
    if not l.strip() or l.startswith("*") or l.startswith("@"):
        break
    f = l.split()
    soil10.append({
        "soil_id": "IBSB910015",
        "salb": float(p[1]), "slu1": float(p[2]), "sldr": float(p[3]), "slro": float(p[4]),
        "slnf": float(p[5]), "slpf": float(p[6]),
        "slb": float(f[0]), "slll": float(f[2]), "sdul": float(f[3]), "ssat": float(f[4]),
        "srgf": float(f[5]), "ssks": float(f[6]), "sbdm": float(f[7]), "sloc": float(f[8]),
        "slcl": float(f[9]), "slsi": float(f[10]), "slhw": float(f[13]),
    })


Build a FileX template dictionary for an annual maize-wheat rotation with two fallow periods:

In [ ]:
rotation10 = {
    "treatment_name": "Maize-wheat rotation",
    "rotation": [
        {"crop": "maize", "cultivar": {"code": "IB0035"},
         "planting": {"date": "1978-03-15", "method": "S", "distribution": "R",
                      "population": 7.2, "row_spacing": 75, "depth": 5}},
        {"crop": "fallow", "end_date": "1978-11-14"},
        {"crop": "wheat", "cultivar": {"code": "IB1500"},
         "planting": {"date": "1978-11-15", "method": "S", "distribution": "R",
                      "population": 7.2, "row_spacing": 75, "depth": 5}},
        {"crop": "fallow", "end_date": "1979-03-14"},
    ],
}


Check inputs before running. The report identifies treatment 1 as a sequence of 4 rotation components:

In [ ]:
sim10 = dl.Simulation(filex_template=rotation10, weather=weather8, soil=soil10)
sim10.check()


Run the sequence simulation for one cycle (the default cycle length is 1 year):

In [ ]:
result10 = sim10.run()


Inspect the summary results for the single cycle, showing rotation component (`R#`), crop (`CR`), and yield (`HWAM`):

In [ ]:
pd.DataFrame(result10.summary())[["R#", "CR", "HWAM"]]


Summarise yield statistics per rotation component with `summarize_seasons`:

In [ ]:
dl.to_dataframe(dl.summarize_seasons(result10.summary()))


Now run the same rotation over 3 cycles (12 component runs) using experiment data controls `years`:

In [ ]:
sim10_3 = dl.Simulation(
    filex_template=rotation10,
    weather=weather8,
    soil=soil10,
    management={"treatments": {1: {"controls": {"years": 3}}}},
)
sim10_3.check()
result10_3 = sim10_3.run()


Inspect the summary results across the 3 cycles:

In [ ]:
pd.DataFrame(result10_3.summary())[["R#", "CR", "HWAM"]]


Summarise yield statistics per rotation component across the 3 cycles with `summarize_seasons`:

In [ ]:
dl.to_dataframe(dl.summarize_seasons(result10_3.summary()))


## Case 11: experiment data per rotation component

Reuse Case 10's rotation template, weather and soil; R 1 is maize and R 3 is wheat, matching the template list and Summary `R#`.

Give maize two 60 kg N/ha events and two 25 mm irrigations, and wheat 40 kg N/ha, repeated over three cycles:

In [ ]:
experiment11 = {"treatments": {1: {
    "controls": {"years": 3},
    "rotation": {
        1: {
            "fertilizer": [
                {"date": "1978-03-15", "material": "FE005", "application": "AP001", "depth": 5, "n": 60},
                {"date": "1978-04-20", "material": "FE005", "application": "AP001", "depth": 5, "n": 60},
            ],
            "irrigation": [
                {"date": "1978-05-01", "amount": 25, "method": "IR001"},
                {"date": "1978-06-01", "amount": 25, "method": "IR001"},
            ],
        },
        3: {"fertilizer": [
            {"date": "1978-11-15", "material": "FE005", "application": "AP001", "depth": 5, "n": 40},
        ]},
    },
}}}


Create a Simulation with the component experiment data; other components and shared levels stay unchanged:

In [ ]:
sim11 = dl.Simulation(
    filex_template=rotation10,
    weather=weather8,
    soil=soil10,
    management=experiment11,
)


Check component periods before running; events after weather-dependent maturity cannot be checked, so leave a margin before the crop ends:

In [ ]:
sim11.check()


Run the sequence with the same component edits in every cycle:

In [ ]:
result11 = sim11.run()


Inspect nitrogen applied (`NICM`, kg N/ha), irrigation applied (`IRCM`, mm) and yield (`HWAM`):

In [ ]:
pd.DataFrame(result11.summary())[["R#", "CR", "NICM", "IRCM", "HWAM"]]


The real DSSAT probe returned NICM 120 and IRCM 50 on every maize row, and NICM 40 on every wheat row; summarise yields per component:

In [ ]:
dl.to_dataframe(dl.summarize_seasons(result11.summary()))


## Case 12: a sweep over planting date and nitrogen

Reuse Case 1's UFGA8201 FileX and Case 8's weather reader for Gainesville's 1982 weather:

In [ ]:
weather12 = weather_rows("UFGA8201")


Define complete planting sections and fertilizer event lists for two dates and 0, 60 or 120 kg N/ha. Split each nitrogen rate into two equal side-dress applications on April 7 and May 17 (FE001 = ammonium nitrate); these event dates stay the same for both planting dates:

In [ ]:
event12 = {"material": "FE001", "application": "AP001", "depth": 10}
planting12 = {"method": "S", "distribution": "R", "population": 7.2, "row_spacing": 61, "depth": 5}
factors12 = {
    "planting": {d: dict(planting12, date=d) for d in ("1982-02-26", "1982-03-12")},
    "fertilizer": {n: [dict(event12, date=d, n=n // 2) for d in ("1982-04-07", "1982-05-17")] if n else [] for n in (0, 60, 120)},
}


Run treatment 1 with the unchanged base first, then all six combinations; short joined labels also become the FileX treatment names:

In [ ]:
rows12 = dl.run_sweep(
    "case1_gainesville/UFGA8201.MZX", weather12, treatments=[1], factors=factors12,
)


Compare factor labels, nitrogen applied (NICM, kg N/ha) and grain yield (HWAM, kg/ha); base factor labels are None and numeric labels stay numbers:

In [ ]:
dl.to_dataframe(rows12)[["scenario", "planting", "fertilizer", "NICM", "HWAM"]]


## Case 13: a cultivar coefficient sweep

Reuse Case 12's FileX and weather for treatment 1, whose source cultivar is IB0035; vary P1 with complete cultivar sections:

In [ ]:
factors13 = {"cultivar": {p1: {"crop": "MZ", "code": "IB0035", "coefficients": {"P1": p1}} for p1 in (200, 259, 320)}}


Run the unchanged base first, then three P1 values; each coefficient run has its own simulation folder with a changed .CUL copy and the source stays unchanged:

In [ ]:
rows13 = dl.run_sweep("case1_gainesville/UFGA8201.MZX", weather12, treatments=[1], factors=factors13)


Compare the P1 label (cultivar), anthesis (ADAT), maturity (MDAT) and grain yield (HWAM, kg/ha); the base label is None:

In [ ]:
dl.to_dataframe(rows13)[["scenario", "cultivar", "ADAT", "MDAT", "HWAM"]]


## Case 14: photosynthesis options and initial conditions off

Reuse Case 12's FileX and weather. Compare DSSAT PHOTO codes `"C"` and `"L"` with a whole `controls` factor. Omitted options keep the FileX value; dssatlab checks the codes, and DSSAT decides what each option does.

Set `initial_conditions: "off"` for every run. The quoted string sets IC to 0 in each FileX copy, so DSSAT supplies initial soil water and nitrogen. Omitting the section keeps the FileX's initial conditions instead; weather and management date checks still apply.

In [ ]:
rows14 = dl.run_sweep(
    "case1_gainesville/UFGA8201.MZX", weather12, treatments=[1],
    management={"treatments": {1: {"initial_conditions": "off"}}},
    factors={"controls": {
        "C": {"photosynthesis": "C"},
        "L": {"photosynthesis": "L"},
    }},
)


The base runs first, followed by C and L. Each factor replaces the whole controls section; initial conditions stay off. Compare grain yield (`HWAM`, kg/ha):

In [ ]:
dl.to_dataframe(rows14)[["scenario", "controls", "HWAM"]]


To give initial conditions yourself, use a dict with `date` and `layers`, as in Case 2. Optional details beside them are `root_mass` (ICRT), `nodule_mass` (ICND), `rhizobia_number` (ICRN), `rhizobia_effectiveness` (ICRE), `residue_n` (ICREN), `residue_p` (ICREP), `residue_incorporation` (ICRIP), and `residue_depth` (ICRID); omitted details write -99. See the [experiment data guide](https://abdelrahmanamr3.github.io/dssatlab/guide/experiment/) for all option codes, IC ranges and units.